# 🎰 Monte Carlo Simulations & Strategy Exploration

This notebook demonstrates long-term lottery outcomes through Monte Carlo simulation.

## Topics Covered
- Understanding expected value
- Simulating thousands of lottery plays
- Comparing different "strategies" (educational only)
- Visualizing the house edge

## ⚠️ Critical Disclaimer
- **Simulations cannot predict lottery outcomes**
- **No strategy can improve your odds**
- **The house edge is mathematically guaranteed**
- This is for educational entertainment only

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.insert(0, '..')

from src.analytics.probability import ProbabilityCalculator
from src.analytics.monte_carlo import MonteCarloSimulator
from src.analytics.strategy import StrategyExplorer

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)  # For reproducibility
%matplotlib inline

# Initialize
prob_calc = ProbabilityCalculator()
simulator = MonteCarloSimulator(seed=42)
strategy_explorer = StrategyExplorer('data/lottery_star.db')

print("Modules loaded!")

## 1. Understanding Lottery Probabilities

In [ ]:
# Compare jackpot odds across games
comparison = prob_calc.compare_games()
df = pd.DataFrame(comparison)
df = df[df['game'].isin(['Mega Millions', 'Powerball', 'Take 5', 'Cash4Life', 'NY Lotto'])]

print("Lottery Jackpot Odds Comparison")
print("=" * 60)
for _, row in df.iterrows():
    print(f"{row['game']:15} - {row['jackpot_odds_str']}")

In [ ]:
# Visualize odds (log scale)
plt.figure(figsize=(12, 6))
bars = plt.barh(df['game'], df['jackpot_odds'], color=sns.color_palette('viridis', len(df)))
plt.xscale('log')
plt.xlabel('Odds (1 in X) - Log Scale')
plt.title('Jackpot Odds Comparison (Lower = Better Odds)')

# Add labels
for bar, odds in zip(bars, df['jackpot_odds_str']):
    plt.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2, 
             odds, va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 Note: Even 'better' odds like Take 5 (1 in 575,757) are still very unlikely!")

## 2. Expected Value Analysis

In [ ]:
# Calculate expected value for each game
print("Expected Value Analysis")
print("=" * 70)
print(f"{'Game':<15} {'Ticket':<10} {'EV':<12} {'Loss/Ticket':<15} {'Return %'}")
print("-" * 70)

for game in ['mega_millions', 'powerball', 'take5', 'ny_lotto']:
    ev = prob_calc.calculate_expected_value(game)
    print(f"{game:<15} ${ev['ticket_cost']:<9.2f} ${ev['expected_value']:<11.4f} ${ev['expected_loss']:<14.4f} {ev['return_percentage']:.2f}%")

print("\n⚠️ The expected loss shows the house edge - you lose money on average!")

In [ ]:
# How jackpot size affects expected value
jackpots = [20, 50, 100, 200, 500, 1000, 1500, 2000]  # Millions

fig, ax = plt.subplots(figsize=(10, 6))

for game, color in [('mega_millions', 'blue'), ('powerball', 'red')]:
    evs = []
    for jp in jackpots:
        ev_data = prob_calc.calculate_expected_value(game, jp)
        evs.append(ev_data['expected_value'])
    
    ax.plot(jackpots, evs, marker='o', label=game.replace('_', ' ').title(), color=color)

ax.axhline(2.0, color='green', linestyle='--', label='Ticket Cost ($2)')
ax.set_xlabel('Jackpot Size (Millions $)')
ax.set_ylabel('Expected Value ($)')
ax.set_title('Expected Value vs Jackpot Size')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Even at $2 billion jackpot, EV is still < ticket cost due to taxes and split probability!")

## 3. Monte Carlo Simulation: Lifetime of Play

In [ ]:
# Simulate playing Powerball for 20 years
# 2 tickets per week = 104 tickets/year = 2080 tickets over 20 years

tickets_per_sim = 100  # tickets per simulation run
num_sims = 10000

result = simulator.simulate_plays('powerball', num_tickets=tickets_per_sim, num_simulations=num_sims)

print("Powerball Simulation Results")
print("=" * 50)
print(f"Total simulated plays: {result.simulations:,}")
print(f"Total spent: ${result.total_spent:,.2f}")
print(f"Total won: ${result.total_won:,.2f}")
print(f"Net result: ${result.net_result:,.2f}")
print(f"ROI: {result.roi_percentage:.2f}%")
print(f"\nJackpot wins: {result.jackpot_wins}")
print(f"\nWins by tier: {result.wins_by_tier}")

In [ ]:
# Visualize win distribution
if result.wins_by_tier:
    wins_df = pd.DataFrame([
        {'tier': k, 'wins': v} 
        for k, v in sorted(result.wins_by_tier.items(), key=lambda x: -x[1])
    ])
    
    plt.figure(figsize=(10, 5))
    plt.bar(wins_df['tier'], wins_df['wins'], color='steelblue')
    plt.xlabel('Match Level (Main+Bonus)')
    plt.ylabel('Number of Wins')
    plt.title(f'Win Distribution ({result.simulations:,} plays)')
    plt.tight_layout()
    plt.show()

## 4. ROI Convergence Over Many Plays

In [ ]:
# Show how ROI converges to expected negative value
conv_df = simulator.run_convergence_test('powerball', max_simulations=50000, check_points=25)

plt.figure(figsize=(12, 6))
plt.plot(conv_df['simulations'], conv_df['roi_pct'], marker='o', linewidth=2)
plt.axhline(0, color='green', linestyle='--', label='Break Even')
plt.axhline(-50, color='red', linestyle='--', alpha=0.5, label='Typical Long-term ROI')
plt.fill_between(conv_df['simulations'], conv_df['roi_pct'], 0, 
                 where=(conv_df['roi_pct'] < 0), alpha=0.3, color='red')
plt.xlabel('Number of Plays')
plt.ylabel('ROI (%)')
plt.title('ROI Convergence Over Time (Powerball)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Notice: ROI consistently stays negative and converges toward expected loss.")
print("   Short-term variance can create illusions of 'winning strategies'.")

## 5. Weekly Play Simulation Over Years

In [ ]:
# Simulate weekly play for 10 years
weekly_df = simulator.simulate_weekly_play('powerball', tickets_per_week=5, years=10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative spending vs winnings
ax1 = axes[0, 0]
ax1.plot(weekly_df['week'], weekly_df['cumulative_spent'], label='Spent', color='red')
ax1.plot(weekly_df['week'], weekly_df['cumulative_won'], label='Won', color='green')
ax1.set_xlabel('Week')
ax1.set_ylabel('Cumulative $')
ax1.set_title('Cumulative Spending vs Winnings')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Net result over time
ax2 = axes[0, 1]
ax2.plot(weekly_df['week'], weekly_df['cumulative_net'], color='purple')
ax2.axhline(0, color='gray', linestyle='--')
ax2.fill_between(weekly_df['week'], weekly_df['cumulative_net'], 0,
                 where=(weekly_df['cumulative_net'] < 0), alpha=0.3, color='red')
ax2.set_xlabel('Week')
ax2.set_ylabel('Net Result ($)')
ax2.set_title('Cumulative Net Result')
ax2.grid(True, alpha=0.3)

# ROI over time
ax3 = axes[1, 0]
ax3.plot(weekly_df['week'], weekly_df['roi_pct'], color='blue')
ax3.axhline(0, color='green', linestyle='--')
ax3.set_xlabel('Week')
ax3.set_ylabel('ROI %')
ax3.set_title('ROI Over Time')
ax3.grid(True, alpha=0.3)

# Weekly wins histogram
ax4 = axes[1, 1]
ax4.hist(weekly_df['weekly_won'], bins=30, color='teal', edgecolor='black', alpha=0.7)
ax4.axvline(weekly_df['weekly_won'].mean(), color='red', linestyle='--', 
            label=f"Mean: ${weekly_df['weekly_won'].mean():.2f}")
ax4.set_xlabel('Weekly Winnings ($)')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Weekly Winnings')
ax4.legend()

plt.suptitle('10 Years of Weekly Powerball Play (5 tickets/week)', fontsize=14)
plt.tight_layout()
plt.show()

final_net = weekly_df['cumulative_net'].iloc[-1]
final_spent = weekly_df['cumulative_spent'].iloc[-1]
print(f"\nFinal Statistics after 10 years:")
print(f"  Total spent: ${final_spent:,.2f}")
print(f"  Net result: ${final_net:,.2f}")

## 6. Strategy Comparison (Educational)

In [ ]:
# Compare different "strategies" - all should perform equally!
print(strategy_explorer.get_strategy_summary())

In [ ]:
# Run strategy comparison simulation
comparison = strategy_explorer.compare_strategies(
    game_name='powerball',
    num_simulations=20000,
    pool_size=69,
    pick_count=5
)

print("Strategy Comparison Results")
print("=" * 70)
print(comparison[['strategy', '0_matches_pct', '1_matches_pct', '2_matches_pct', '3_matches_pct']].to_string(index=False))

print("\n⚠️ All strategies should show similar results - any differences are random variance!")

In [ ]:
# Visualize strategy comparison
strategies = comparison['strategy']
match_cols = [f'{i}_matches_pct' for i in range(4)]

x = np.arange(len(strategies))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))

for i, col in enumerate(match_cols):
    ax.bar(x + i*width, comparison[col], width, label=f'{i} matches', alpha=0.8)

ax.set_ylabel('Percentage (%)')
ax.set_title('Strategy Performance Comparison (Should Be Similar)')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(strategies, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✅ As expected, all strategies perform similarly - proving no strategy has an advantage!")

## Conclusions

### What We Learned

1. **The House Always Wins**: Expected value is always less than ticket cost. This is by design.

2. **Variance Creates Illusions**: Short-term wins can make any "strategy" seem effective. Long-term, all strategies lose equally.

3. **No Predictive Patterns**: Hot numbers, cold numbers, pairs, trends - none provide predictive value.

4. **Scale of Odds**: Jackpot odds are astronomical. Even playing for a lifetime, the probability of winning remains negligible.

### The Only "Strategy" That Works

- **Set a budget**: Only spend what you can afford to lose
- **Treat it as entertainment**: Like buying a movie ticket
- **Never chase losses**: The math doesn't change
- **Understand the odds**: 1 in 292 million is effectively zero

### Final Thought

The lottery is a form of entertainment that comes with a cost. If you play, enjoy the momentary excitement, but understand the mathematical reality: **you are paying for hope, not a realistic chance of winning.**

*Play responsibly. This analysis is for educational purposes only.*